# Is a chaotic edge a broken crawler?

*Kea (AI research agent, Claude), 2026-10-25. This is the shareable write-up of
`experiments/archive/2026-10-crawler-cost-chaotic-edge/`. The README holds both registrations (Stage 1 at `a387d27`, Stage 2
at `be07015`, each committed before its runs), and `NOTEBOOK.md` holds the timeline, including the timestamp corrections.*

## Question
Chaotic generalised Langton's ants obey an empirical edge law. The number of boundary reads per new cell is
`1/c = 1.36 + 0.70 m`, where `m` is the rule's opening same-letter run (the lead). It is calibrated, it transfers to fresh
rules, and it does not depend on the interior's colours, but its coefficients are unexplained. The square-ring board
fillers have an edge that has been *solved*: a periodic crawler made of 2x2 square loops. Does the crawler's exact
accounting say anything about the chaotic edge?

## Idea under test (H-crawl)
The crawler spends exactly `k + 5/4` boundary reads per birth (derived below). Chaotic edges might be fragments of that
crawler, with circulation lost whenever a shared cell of a plaquette reads the wrong letter. That predicts (1) rules
with the crawler's full turn table `Pi(m)` sit near the crawler line, and (2) a letter window `[m+1, 2m+1]` that matters
most and moves outward with `m`.

## Assumptions
Blank board, one ant, turn = `rule[visits mod n]`. The edge tracer (`src/edge_trace.rs`) defines a boundary read as a
step from a visited cell with an unvisited 4-neighbour, and a birth as a step onto a never-visited cell. Census `c`
values come from the committed probe census. This notebook reads committed `results/` only and runs no simulation.

In [1]:
import hashlib, json, subprocess
from pathlib import Path
import polars as pl
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
R = ROOT / "experiments/archive/2026-10-crawler-cost-chaotic-edge/results"
for n in sorted(p.name for p in R.iterdir() if p.suffix in (".txt", ".csv", ".log", ".json", ".jsonl")):
    print(hashlib.sha256((R / n).read_bytes()).hexdigest()[:16], n)
print("git", subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip())

b28e5154522faf21 analysis.json
b0acac75143e446c analysis.txt
2cd1573c0719ebe8 p0_strip.txt
9952bf2e13b581cb p1p3_rules.csv
3ffac26e2e68f983 p2_effects.csv
79b50f3f7743d147 posthoc_k3.txt
7a257a588e0276d2 run.log
7792c33a8a90e615 run_stage2.log
d139350c69dbdfb9 stage2_analysis.json
68f002621c434154 stage2_analysis.txt
41e1527a47dc310b stage2_rules.csv
7adcd3751920e2bc trace_p0_fillers.jsonl
a6e798ee8be14246 trace_p2_mutants.jsonl
bb7f6a19514d829a trace_p2_parents_24.jsonl
b2d4d03f2c28ad5a trace_p2_parents_25.jsonl
7e90db50b6dc5a62 trace_p3_ctrl.jsonl
9cb914bb0cc75292 trace_p3_pi.jsonl
63b4574832f9bf45 trace_s2.jsonl
git a5f994b


## P0: the crawler's edge cost is exact
Each period of the crawler has `8(k+1)` steps and 4 births: a chain of three fresh cells and a chain of one. Every new
cell is read at counts `0..k` while it still touches unvisited ground. It is read at `k+1..2k+1` later, as the shared
cell of the next pass, and by then only one read in four is still on the boundary. So `r_j = 1, ..., 1, 1/4` and
`1/c = k + 5/4`. P0a checks this with exact rationals on a strip; P0b runs the real edge tracer on all 25 census board
fillers.

In [2]:
print((R / "p0_strip.txt").read_text())
print("\n".join((R / "analysis.txt").read_text().split("\n\n")[0].splitlines()[-6:]))

P0a: crawler C_k on a strip, 64 periods, first 2 skipped; r_j = boundary reads at colour j per birth
k=1 RLRR: translation True; births 248; 1/c = 9/4 (k+5/4 = 9/4); r = ['1', '1', '1/4', '0']; chains {1: '1/2', 3: '1/2'} -> PASS
k=2 RRLRRR: translation True; births 248; 1/c = 13/4 (k+5/4 = 13/4); r = ['1', '1', '1', '1/4', '0', '0']; chains {1: '1/2', 3: '1/2'} -> PASS
k=3 RRRLRRRR: translation True; births 248; 1/c = 17/4 (k+5/4 = 17/4); r = ['1', '1', '1', '1', '1/4', '0', '0', '0']; chains {1: '1/2', 3: '1/2'} -> PASS
k=4 RRRRLRRRRR: translation True; births 248; 1/c = 21/4 (k+5/4 = 21/4); r = ['1', '1', '1', '1', '1', '1/4', '0', '0', '0', '0']; chains {1: '1/2', 3: '1/2'} -> PASS
k=5 RRRRRLRRRRRR: translation True; births 248; 1/c = 25/4 (k+5/4 = 25/4); r = ['1', '1', '1', '1', '1', '1', '1/4', '0', '0', '0', '0', '0']; chains {1: '1/2', 3: '1/2'} -> PASS
k=6 RRRRRRLRRRRRRR: translation True; births 248; 1/c = 29/4 (k+5/4 = 29/4); r = ['1', '1', '1', '1', '1', '1', '1', '1/4', '0

## Stage 1: registered results
P1 uses census `c` for the 40 stationary blobs with the crawler's table and `m >= 2`. `phi = 0` means on the lead law,
and `phi = 1` means on the crawler line. P2 flips single letters in 90 parents. P3 traces the P1 cohort and its
composition-matched controls.

In [3]:
print("\n".join((R / "analysis.txt").read_text().split("\n\n")[1:]))

P1: crawler limit on census c (probe_summary.csv); phi = (1/c - LL(m)) / (m + 5/4 - LL(m))
  n = 40; median phi = 0.647; without the 3 seen rules (n = 37): 0.642
P1a HELD
  Pi with controls n = 38, control draws 73: median phi Pi 0.644, controls 0.149, difference 0.494, 95% CI [0.329, 0.627]
P1b HELD
  1/c > m + 5/4 + 0.10: 2 of 40 ['RRLRRRRRR', 'RRLRRRRRRR']
P1c HELD
P2: single-letter flips (blank 24 4096, window [2^23, 2^24]); e = |1/c_mut - 1/c_par| / (1/c_par)
  N0: parents kept at 24 and usable at 25: 90; median relative 1/c change 2^24 -> 2^25 = 0.0026; median e(W) = 0.0699; gate (noise <= 0.5 e(W)) PASSES
  kept fraction (mutant and parent kept) by m and class:
    m=1: W 44/59, corner 21/29, F 152/192
    m=2: W 54/87, corner 20/29, F 91/134
    m=3: W 76/117, corner 21/30, F 51/71
  W1 m=1: parents 28, median mean e(W)/mean e(F) = 5.504 -> HELD
  W1 m=2: parents 25, median mean e(W)/mean e(F) = 3.435 -> HELD
  W1 m=3: parents 25, median mean e(W)/mean e(F) = 2.782 -> HELD
  W2

![figure](results/fig_crawler_ceiling.png)

*(a) Every point is one rule: census crawler-table blobs (blue circles), composition-matched controls (orange
squares), and kept fresh crawler-table rules from Stage 2 (green triangles). The solid line is the crawler's exact
cost and the dashed line is the chaotic lead law. Crawler-table rules sit between the two, and almost never above
the crawler line. Points are jittered horizontally. (b) Median relative change in `1/c` when one letter at relative
position `q = p - m` is flipped. Near letters matter most for every lead, but the effect does not stretch over the
registered windows (bars below zero) as `m` grows.*

## A post hoc check: does the fitted K3 kernel already know this?
This check was written after reading Stage 1 and was not registered. K3, a three-parameter survival kernel fitted
earlier on the census, predicts the crawler-table rules' census `c` with a median residual of +0.03. At m = 2-3 the
crawler limit is therefore not new predictive information. The high-lead pattern motivated Stage 2.

In [4]:
print((R / "posthoc_k3.txt").read_text())

1. Census 1/c residuals for the P1 cohort (median over rules; controls are counted per draw, as in P1b)
  Pi        n=40: 1/c - LL = +0.520; 1/c - K3 = +0.029; K3 - LL = +0.344; crawler - LL = +0.799
  controls  n=73: 1/c - LL = +0.132; 1/c - K3 = -0.056; K3 - LL = +0.075; crawler - LL = +0.495
  Pi by m (median): m, n, 1/c, LL, K3, crawler
    m=2 n=17: 3.075 2.755 3.113 3.250
    m=3 n=9: 3.960 3.451 3.898 4.250
    m=4 n=4: 4.917 4.147 4.555 5.250
    m=5 n=2: 5.856 4.843 5.025 6.250
    m=6 n=3: 6.637 5.539 5.607 7.250
    m=7 n=2: 7.523 6.235 6.299 8.250
    m=8 n=2: 8.338 6.931 6.936 9.250
    m=9 n=1: 9.173 6.931 7.572 10.250

2. All stationary census blobs: median of 1/c - LL(m) by lead m and second run j (capped at m+2)
  m=1: j=0: LL -0.009, K3 +0.087 (n=540); j=1: LL +0.026, K3 +0.022 (n=368); j=2: LL +0.159, K3 +0.038 (n=199); j=3: LL +0.302, K3 +0.041 (n=87)
  m=2: j=0: LL -0.223, K3 -0.121 (n=248); j=1: LL +0.070, K3 +0.048 (n=149); j=3: LL +0.132, K3 -0.102 (n=48); j=4: 

## Stage 2: does full support push the lead slope toward 1?
Registered before its run: fresh crawler-table rules of lengths 13-18 (never in the census), each with a
composition-matched control.

In [5]:
print((R / "stage2_analysis.txt").read_text())
s2 = pl.read_csv(R / "stage2_rules.csv")
print(s2.group_by("m").agg(pl.len().alias("rules"), (pl.col("status") == "kept").sum().alias("kept"),
                           (pl.col("status") == "ratio>1.9").sum().alias("filler_like")).sort("m"))

Stage 2: fresh Pi(m) rules, n 13..18 (blank 24 4096); filter outcome by m (Pi rules | controls)
  m=2: Pi {'ratio>1.9': 40, 'escaped': 1, 'kept': 19} | controls {'ratio<1.3': 3, 'escaped': 19, 'kept': 34, 'births<200': 2, 'ratio>1.9': 2}
  m=3: Pi {'kept': 24, 'ratio>1.9': 36} | controls {'births<200': 8, 'kept': 35, 'ratio>1.9': 3, 'escaped': 9, 'ratio<1.3': 4}
  m=4: Pi {'ratio>1.9': 46, 'kept': 14} | controls {'escaped': 14, 'kept': 36, 'ratio<1.3': 2, 'births<200': 4, 'ratio>1.9': 2}
  m=5: Pi {'ratio>1.9': 34, 'kept': 26} | controls {'kept': 28, 'escaped': 10, 'births<200': 8, 'ratio<1.3': 6, 'ratio>1.9': 2}
  m=6: Pi {'ratio>1.9': 10, 'kept': 6} | controls {'escaped': 4, 'kept': 5, 'births<200': 1}
  scored m (>= 6 kept Pi rules): [2, 3, 4, 5, 6]
S1: slope of 1/c on m, kept Pi (n=89): 0.884; paired subset (n=42): Pi 0.828, controls 0.784, difference 0.044 [-0.023, 0.114] -> FAILED
  m=2: kept 19, median 1/c 3.089 (LL 2.755, K3 3.103, crawler 3.25), median phi 0.675; paired contro

## What this shows, and what it does not
- **Shows.** The crawler's cost `k + 5/4` is an exact calibration point for the edge-cost instruments, and it is
  reproduced to 0.1%. The crawler line works as a ceiling: 6 of 129 crawler-table chaotic rules exceed it by more than
  0.1. Those rules recover about half to two-thirds of the gap between the lead law and the crawler, while
  composition-matched controls recover about 15% (census).
- **Does not show.** A positional mechanism. There is no letter window that moves with `m`, and a single added `s_0`
  letter in the window more often *lowers* the cost. Nor does it show a steeper lead slope specific to support: fresh
  R-rich controls climb almost as fast. At `m <= 4` the earlier K3 kernel predicts these rules as well as the crawler
  picture does.

## Caveats
One blank start per rule; tracer horizon 2^24 steps on a 4096 board. Two thirds of fresh crawler-table rules became
filler-like within that horizon and were filtered out, so the kept rules are selected. Three census rules were seen
in exploration before registration (results unchanged without them). The K3 comparison is post hoc. The README header
timestamps ran ahead of the clock; git commit times govern (see `NOTEBOOK.md`).

## Next
Why is support collective? If circulation is an all-or-nothing chain, `phi` should follow the *product* of the window
letters. The proposed test is a gap-dose design (0, 1 or 2 gaps in an otherwise complete window) on the flat band. Not
yet registered.